# Regressione polinomiale

L'obiettivo di questo notebook è mostrare come approssimare una funzione tramite una combinazione lineare di polinomi (fino al grado $D$):

$$y=f(x)\approx\sum_{m=0}^Dw_m\varphi_m(x)$$

Dove $\varphi_n$ è il polinomio di base di grado $m$.

In [ ]:
# Importiamo i moduli Python basilari
import numpy as np
import matplotlib.pyplot as plt

## Creazione del dataset

Creiamo un dataset partendo da una funzione non-algebrica (non polinomiale) definita per comodità in $[0,1]\mapsto[0,1]$.

In [ ]:
true_fun  = lambda x: np.sin(2*np.pi*x)*np.exp(-x)*0.5+0.5

N = 50
x_data = np.linspace(0.0,1.0,N)
y_data = true_fun(x_data)

# Aggiungo del rumore gaussiano
std_noise = 0.05
y_noise = np.random.normal(loc=0.0,scale=std_noise,size=N)
y_data += y_noise

Usiamo una funzione di `sklearn` per suddividere il dataset in training set e test set in modo automatico.

In [ ]:
from sklearn.model_selection import train_test_split

# Divido: 67% train vs 33% test
split = 0.33
seed = 42

x_train, x_test, y_train, y_test = \
    train_test_split(x_data, y_data, test_size=split, random_state=seed)

N_train = len(x_train)
print("N_train =",N_train)
N_test = len(x_test)
print("N_test =",N_test)

Visualizziamo la funzione da approssimare, usando colori diversi per training set e test set.

In [ ]:
%matplotlib ipympl

fig1, ax1 = plt.subplots()
plt.plot(x_train,y_train,'r.',label='training set')
plt.plot(x_test,y_test,'bx',label='test set')
plt.legend()
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## Equazioni ai minimi quadrati

Breve recap sull'equazione ai minimi quadrati - partiamo da un sistema **sovradeterminato** (più equazioni che parametri):

$$\begin{cases}
w_0 + w_1\varphi_1(x_1)+\dots+w_D\varphi_D(x_1) = y_1 \\
w_0 + w_1\varphi_1(x_2)+\dots+w_D\varphi_D(x_2) = y_2 \\
\dots \\
w_0 + w_1\varphi_1(x_{N_{\text{TR}}})+\dots+w_D\varphi_D(x_{N_{\text{TR}}}) = y_N \\
\end{cases} \quad \implies X\boldsymbol{w}=\boldsymbol{y} $$

Dove:

$$\quad X = \begin{bmatrix} 1 & \varphi_1(x_1) & \dots & \varphi_D(x_1) \\\ 
1 & \varphi_1(x_2) & \dots & \varphi_D(x_2) \\\
\dots & \dots & \dots & \dots\\\
1 & \varphi_1(x_{N_{\text{TR}}}) & \dots & \varphi_D(x_{N_{\text{TR}}}) \end{bmatrix} \; , \quad \boldsymbol{y} = \begin{bmatrix} y_1 \\\ 
y_2 \\\
\dots \\\
y_{N_{\text{TR}}} \end{bmatrix}$$

Costruiamo la matrice $X$ usando polinomi di Legendre:

$$\varphi_m(x) = \frac{1}{2^mm!}\frac{d^m}{dx^m}\big(x^2-1\big)^m$$

Scegliamo i polinomi di Legendre poiché formano una base ortonormale, ovvero:

$$\int_{-1}^1 \varphi_k(x)\varphi_l(x)dx = \begin{cases} 1 & k=l\\
0 & k \ne l\end{cases}$$

<span style="color:red;">**Provate a cambiare l'ordine** $D$ **del polinomio (in modo da ossevare underfitting e overfitting)!**</span>

In [ ]:
from scipy.special import legendre

# Piccolo cambio di coordinate per definire i polinomi tra [0,1] e mantenere l'ortonormalita'
legendre_d = lambda x, d : np.sqrt(2*d+1)*(legendre(d))(2*x-1)

# Ordine del polinomio (+1)
D = 5

X = np.ones((N_train,1))
for d in range(1,D) :
    x_d = legendre_d(x_train,d)
    x_d = x_d.reshape(-1, 1)
    X = np.concatenate((X, x_d), axis=1)
    
# Dimensioni della matrice X
print("X.shape =",X.shape)

Il sistema ai minimi quadrati:

$$ \frac{1}{N_{\text{TR}}}X^TX\boldsymbol{w}=\frac{1}{N_{\text{TR}}}X^T\boldsymbol{y}$$

Minimizza la _funzione obiettivo_ ("cost function"), i.e. errore quadratico:

$$\mathcal{L} = \frac{1}{2N_{\text{TR}}}\Big\{\sum_{n=0}^{N_{\text{TR}}}\big[\sum_{m=0}^Dw_m\varphi_m(x_n)-y_n\big]^2\Big\}$$

Notare che l'errore quadratico è normalizzato sul numero di dati nel training set.

In [ ]:
# Equivalente a: np.matmul(X.transpose,X)
S = X.T@X/N_train
print("S.shape =",S.shape)

y_star = X.T@y_train/N_train
print("y_star.shape =",y_star.shape)

Risolviamo il sistema ai minimi quadrati usando un metodo di `numpy.linalg`:

In [ ]:
from numpy.linalg import solve

w_star = solve(S,y_star)

# Soluzione (i.e. vettore dei parametri/pesi)
print(w_star)

## Valutazione e calcolo dell'errore

Usiamo `lambda` per definire il polinomio interpolatore su una singola riga:

In [ ]:
y_model = lambda x, w: sum([w[d]*legendre_d(x,d) for d in range(D)])

In [ ]:
# Punti dove valutare l'interpolatore
N_plot = 500
x_plot = np.linspace(0,1,N_plot)

%matplotlib ipympl
fig2, ax2 = plt.subplots()
plt.plot(x_train,y_train,'r.',label='training set')
plt.plot(x_test,y_test,'bx',label='test set')
plt.plot(x_plot,y_model(x_plot,w_star),'k-',label='model')
plt.legend()
plt.xlim([-0.1,1.1])
plt.ylim([-0.1,1.1])
plt.xlabel("x")
plt.ylabel("y")
plt.show()

Calcoliamo l'errore quadratico medio. Osservate attentamente l'ordine delle operazioni!

In [ ]:
ls_err = np.sqrt(np.sum((y_model(x_test,w_star)-y_test)**2)/N_test)
print("LS error =",ls_err)

Calcoliamo anche la norma quadratica del vettore dei coefficienti, scalata sul grado del polinomio:

In [ ]:
from numpy.linalg import norm

w_norm = norm(w_star,2)/D
print("||w||_2 =",w_norm)

## Errore vs. dimensione del modello

Per ottenere l'andamento dell'errore di training sul numero di parametri ci avvarremo di una funzione ausiliaria `cross_validation()` che implementa un semplice metodo di [**cross validation**](https://en.wikipedia.org/wiki/Cross-validation_(statistics)):

In [ ]:
from bias_variance import cross_validation
help(cross_validation)

Calcoliamo e visualizziamo errore sul training set e sul test fino a un numero massimo di parametri `D_max`:

In [ ]:
D_vec = []
lse_train_vec = []
lse_test_vec = []
wn_vec = []

D_max = 40
for D in range(1,D_max) :
    print("Performing cross-validation for D =",D)
    lse_train, lse_test, wn = cross_validation(x_data, y_data, D)
    D_vec.append(D)
    lse_train_vec.append(lse_train)
    lse_test_vec.append(lse_test)
    wn_vec.append(wn)

In [ ]:
%matplotlib ipympl
fig3, ax3 = plt.subplots()
plt.semilogy(D_vec,lse_train_vec,'r:',label="train error")
plt.semilogy(D_vec,lse_test_vec,'b-',label="test error")
plt.semilogy(D_vec,wn_vec,'k--',label="norm of weights vector")
plt.legend()
plt.xlabel("Polynomial degree (D)")
plt.ylabel("Relative errors")
plt.show()

<span style="color:red;">**Provate ad aumentare** `D_max` **ben oltre il numero di dati nel training set.**</span>

L'equazione ai minimi quadrati per un sistema **sottodeterminato** (i.e. più parametri che equazioni) conserva ancora la proprietà di unicità, e minimizza la norma della soluzione (i.e. vettore dei parametri). Questo porta l'errore sul test set a diminuire nuovamente. Questo fenomeno è chiamato _doppia discesa_ (double descent), e si specula sia molto importante per l'allenamento delle reti neurali (tema controverso).

Un articolo sul tema: https://iopscience.iop.org/article/10.1088/1742-5468/ac3a74

Un video YouTube: https://www.youtube.com/watch?v=z64a7USuGX0

# Regolarizzazione

## Ridge regression

Possiamo aggiungere un termine di penalizzazione all'errore di training per regolarizzare la regressione. Se il termine è quadratico, si parla di _ridge regression_:

$$ \mathcal{L} = \frac{1}{2N_{\text{TR}}}\Big\{\sum_{n=0}^{N_{\text{TR}}}\big[\sum_{m=0}^Dw_m\varphi_m(x_n)-y_n\big]^2\Big\} + \lambda\sum_{m=0}^Dw_m^2 = \frac{1}{2N_{\text{TR}}}||\varphi(\boldsymbol{x})-\boldsymbol{y}||_2^2 + \lambda||\boldsymbol{w}||_2^2$$

È possibile trovare il minimo di $\mathcal{L}$ risolvendo un'equazione ai minimi quadrati:

$$\Big(\frac{1}{N_{\text{TR}}}X^TX+\lambda I\Big)\boldsymbol{w} = \frac{1}{N_{\text{TR}}}X^T\boldsymbol{y}$$

In [ ]:
# Ordine del polinomio (+1), provate altri valori!
D = 20

X = np.ones((N_train,1))
for d in range(1,D) :
    x_d = legendre_d(x_train,d)
    x_d = x_d.reshape(-1, 1)
    X = np.concatenate((X, x_d), axis=1)

print("X.shape =",X.shape)

In [ ]:
# Moltiplicatore del termine di penalizzazione
# Provate a cambiarlo per vedere l'effetto della regolarizzazione
l_ridge = 1e-2

S = X.T@X/N_train
S_ridge = S+l_ridge*np.eye(S.shape[0],S.shape[1])
y_star = X.T@y_train/N_train

w_ridge = solve(S_ridge,y_star)

In [ ]:
ls_err = np.sqrt(np.sum((y_model(x_test,w_ridge)-y_test)**2)/N_test)
print("LS error =",ls_err)

w_norm = norm(w_ridge,2)/D
print("||w||_2 =",w_norm)

In [ ]:
N_plot = 500
x_plot = np.linspace(0,1,N_plot)

%matplotlib ipympl
fig4, ax4 = plt.subplots()
plt.plot(x_train,y_train,'r.',label='training set')
plt.plot(x_test,y_test,'bx',label='test set')
plt.plot(x_plot,y_model(x_plot,w_ridge),'k-',label='model')
plt.legend()
plt.xlim([-0.1,1.1])
plt.ylim([-0.1,1.1])
plt.xlabel("x")
plt.ylabel("y")
plt.show()

## LASSO regression

LASSO = Least Absolute Shrinkage and Selection Operator

Invece di penalizzare con un termine quadratico, aggiungiamo la norma 1 dei pesi:

$$ \mathcal{L} = \frac{1}{2N_{\text{TR}}}\Big\{\sum_{n=0}^{N_{\text{TR}}}\big[\sum_{m=0}^Dw_m\varphi_m(x_n)-y_n\big]^2\Big\} + \lambda\sum_{m=0}^D|w_m| = \frac{1}{2N_{\text{TR}}}||\varphi(\boldsymbol{x})-\boldsymbol{y}||_2^2 + \lambda||\boldsymbol{w}||_1$$

Stavolta non è possibile scrivere un'equazione ai minimi quadrati, quindi ci affidiamo ad un ottimizzatore di `sklearn.linear_model`.

In [ ]:
# Ordine del polinomio (+1), provate altri valori!
D = 20

# Matrice senza intercetta da passare a sklearn.linear_model.Lasso
X_lasso = np.empty((N_train,0))
for d in range(1,D) :
    x_d = legendre_d(x_train,d)
    x_d = x_d.reshape(-1, 1)
    X_lasso = np.concatenate((X_lasso, x_d), axis=1)
# print(X)
print("X_lasso.shape =",X_lasso.shape)

In [ ]:
from sklearn.linear_model import Lasso

l_lasso = 1e-2

clf = Lasso(alpha=l_lasso,max_iter=10000,tol=5e-5)
clf.fit(X_lasso,y_train)
w_lasso = np.concatenate((clf.intercept_,clf.coef_),axis=None)

print("w =",w_lasso)

Osservate quanti pesi sono stati azzerati. Vettori/matrici sparse = speedup computazionale! 

In [ ]:
ls_err = np.sqrt(np.sum((y_model(x_test,w_lasso)-y_test)**2)/N_test)
print("LS error =",ls_err)

w_norm = norm(w_lasso,1)/D
print("||w||_1 =",w_norm)

In [ ]:
N_plot = 500
x_plot = np.linspace(0,1,N_plot)

%matplotlib ipympl
fig5, ax5 = plt.subplots()
plt.plot(x_train,y_train,'r.',label='training set')
plt.plot(x_test,y_test,'bx',label='test set')
plt.plot(x_plot,y_model(x_plot,w_lasso),'k-',label='model')
plt.legend()
plt.xlim([-0.1,1.1])
plt.ylim([-0.1,1.1])
plt.xlabel("x")
plt.ylabel("y")
plt.show()